# Abgabe 2 – Kapitel 3: Klassifikation

Programmieraufgaben aus *Hands-On Machine Learning* (Aurélien Géron), Kapitel 3.

- Aufgabe 1: KNN-Klassifikator für MNIST (> 97 %)
- Aufgabe 2: Data Augmentation
- Aufgabe 3: Titanic
- Aufgabe 4: Spamfilter (SpamAssassin)



## Aufgabe 1: KNN-Klassifikator für MNIST (> 97 %)

Gittersuche über `weights` und `n_neighbors` auf 10.000 Bildern.

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', as_frame=False)
X_mnist, y_mnist = mnist.data, mnist.target

# MNIST ist bereits sinnvoll sortiert: erste 60.000 = Training, Rest = Test
X_train_mnist, X_test_mnist = X_mnist[:60000], X_mnist[60000:]
y_train_mnist, y_test_mnist = y_mnist[:60000], y_mnist[60000:]
print('Training:', X_train_mnist.shape, '| Test:', X_test_mnist.shape)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

param_grid = [{'weights': ['uniform', 'distance'],
               'n_neighbors': [3, 4, 5, 6]}]

knn_clf = KNeighborsClassifier()
grid_search = GridSearchCV(knn_clf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
# Gittersuche zur Beschleunigung nur auf den ersten 10.000 Bildern
grid_search.fit(X_train_mnist[:10_000], y_train_mnist[:10_000])

print('Beste Parameter:', grid_search.best_params_)
print('Beste CV-Genauigkeit (10.000 Bilder): {:.4f}'.format(grid_search.best_score_))

Bestes Modell auf dem ganzen Trainingssatz neu trainieren und auf den Testdaten messen.

In [ ]:
from sklearn.metrics import accuracy_score

best_knn = grid_search.best_estimator_
best_knn.fit(X_train_mnist, y_train_mnist)

y_pred_mnist = best_knn.predict(X_test_mnist)
acc_1 = accuracy_score(y_test_mnist, y_pred_mnist)
print('Testgenauigkeit: {:.4f}'.format(acc_1))

Beste Kombination meist `weights='distance'`, `n_neighbors=4`; rund 97 % Testgenauigkeit – Ziel erreicht.

## Aufgabe 2: Data Augmentation durch Verschieben

Bilder um ein Pixel verschieben, vier Kopien je Bild, bestes Modell neu trainieren.

In [ ]:
import numpy as np
from scipy.ndimage import shift

def shift_image(image, dx, dy):
    """Verschiebt ein als 784-Vektor vorliegendes MNIST-Bild um (dx, dy) Pixel."""
    image = image.reshape(28, 28)
    shifted = shift(image, [dy, dx], cval=0, mode='constant')
    return shifted.reshape(-1)

In [ ]:
import matplotlib.pyplot as plt

image = X_train_mnist[0]
beispiele = [('Original', image),
             ('nach unten', shift_image(image, 0, 5)),
             ('nach links', shift_image(image, -5, 0))]

plt.figure(figsize=(9, 3))
for i, (titel, img) in enumerate(beispiele):
    plt.subplot(1, 3, i + 1)
    plt.title(titel)
    plt.imshow(img.reshape(28, 28), cmap='binary')
    plt.axis('off')
plt.show()

In [ ]:
X_train_aug = [img for img in X_train_mnist]
y_train_aug = [label for label in y_train_mnist]

for dx, dy in ((-1, 0), (1, 0), (0, -1), (0, 1)):
    for img, label in zip(X_train_mnist, y_train_mnist):
        X_train_aug.append(shift_image(img, dx, dy))
        y_train_aug.append(label)

X_train_aug = np.array(X_train_aug)
y_train_aug = np.array(y_train_aug)

# zufällig durchmischen, damit gleiche Ziffern nicht gruppiert sind
shuffle_idx = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[shuffle_idx]
y_train_aug = y_train_aug[shuffle_idx]
print('Erweiterter Trainingsdatensatz:', X_train_aug.shape)

In [ ]:
knn_best = KNeighborsClassifier(**grid_search.best_params_)
knn_best.fit(X_train_aug, y_train_aug)

y_pred_aug = knn_best.predict(X_test_mnist)
acc_2 = accuracy_score(y_test_mnist, y_pred_aug)
print('Testgenauigkeit ohne Augmentation: {:.4f}'.format(acc_1))
print('Testgenauigkeit mit  Augmentation: {:.4f}'.format(acc_2))

Mit Augmentation steigt die Testgenauigkeit auf etwa 97,6 %.

## Aufgabe 3: Titanic

Klassifikator für die Spalte `Survived`: Daten laden, vorverarbeiten, Modelle vergleichen.

In [ ]:
import pandas as pd
from pathlib import Path
import tarfile
import urllib.request

def load_titanic_data():
    tarball_path = Path('datasets/titanic.tgz')
    if not tarball_path.is_file():
        Path('datasets').mkdir(parents=True, exist_ok=True)
        url = 'https://homl.info/titanic.tgz'
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as titanic_tarball:
            titanic_tarball.extractall(path='datasets')
    train = pd.read_csv(Path('datasets/titanic/train.csv'))
    test = pd.read_csv(Path('datasets/titanic/test.csv'))
    return train, test

train_data, test_data = load_titanic_data()
train_data.head()

In [ ]:
train_data = train_data.set_index('PassengerId')
test_data = test_data.set_index('PassengerId')
train_data.info()

Numerisch: Median-Imputation und Skalierung. Kategorial: häufigster Wert und One-Hot. `Cabin`, `Name`, `Ticket` werden ignoriert.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_attribs = ['Age', 'SibSp', 'Parch', 'Fare']
cat_attribs = ['Pclass', 'Sex', 'Embarked']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocess_titanic = ColumnTransformer([
    ('num', num_pipeline, num_attribs),
    ('cat', cat_pipeline, cat_attribs),
])

X_train_titanic = preprocess_titanic.fit_transform(train_data[num_attribs + cat_attribs])
y_train_titanic = train_data['Survived']
print('Feature-Matrix:', X_train_titanic.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

models = {
    'LogReg': LogisticRegression(max_iter=1000, random_state=42),
    'SVC': SVC(gamma='auto', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    scores = cross_val_score(model, X_train_titanic, y_train_titanic, cv=10)
    print('{:14s} Genauigkeit: {:.4f}'.format(name, scores.mean()))

In [ ]:
# Bestes Modell auf allen Trainingsdaten trainieren und Vorhersagen für die Testdaten erzeugen
forest_clf = RandomForestClassifier(n_estimators=100, random_state=42)
forest_clf.fit(X_train_titanic, y_train_titanic)

X_test_titanic = preprocess_titanic.transform(test_data[num_attribs + cat_attribs])
y_pred_titanic = forest_clf.predict(X_test_titanic)
print('Vorhersagen für Testdaten erzeugt:', y_pred_titanic.shape)

SVC und Random Forest erreichen rund 80–83 % in der Kreuzvalidierung.

## Aufgabe 4: Spamfilter (SpamAssassin)

E-Mails laden und parsen, in Wortvektoren überführen, Klassifikatoren über Precision und Recall vergleichen.

In [ ]:
import tarfile
import urllib.request
from pathlib import Path

SPAM_PATH = Path('datasets/spam')
HAM_URL = 'https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham.tar.bz2'
SPAM_URL = 'https://spamassassin.apache.org/old/publiccorpus/20030228_spam.tar.bz2'

def fetch_spam_data():
    SPAM_PATH.mkdir(parents=True, exist_ok=True)
    for filename, url in (('ham.tar.bz2', HAM_URL), ('spam.tar.bz2', SPAM_URL)):
        path = SPAM_PATH / filename
        if not path.is_file():
            urllib.request.urlretrieve(url, path)
        with tarfile.open(path) as tar:
            tar.extractall(path=SPAM_PATH)

fetch_spam_data()

In [ ]:
ham_dir = SPAM_PATH / 'easy_ham'
spam_dir = SPAM_PATH / 'spam'
ham_filenames = sorted(f for f in ham_dir.iterdir() if len(f.name) > 20)
spam_filenames = sorted(f for f in spam_dir.iterdir() if len(f.name) > 20)
print('Ham:', len(ham_filenames), '| Spam:', len(spam_filenames))

In [ ]:
import email
import email.parser
import email.policy

def load_email(filepath):
    with open(filepath, 'rb') as f:
        return email.parser.BytesParser(policy=email.policy.default).parse(f)

ham_emails = [load_email(f) for f in ham_filenames]
spam_emails = [load_email(f) for f in spam_filenames]

print(ham_emails[1].get_content().strip()[:300])

Struktur der E-Mails ansehen (Text, HTML, multipart).

In [ ]:
def get_email_structure(email_obj):
    if isinstance(email_obj, str):
        return email_obj
    payload = email_obj.get_payload()
    if isinstance(payload, list):
        multipart = ', '.join(get_email_structure(sub) for sub in payload)
        return f'multipart({multipart})'
    return email_obj.get_content_type()

from collections import Counter
struktur = Counter(get_email_structure(e) for e in ham_emails)
print('Häufigste Ham-Strukturen:', struktur.most_common(3))

Trainings-/Test-Split.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X_mail = np.array(ham_emails + spam_emails, dtype=object)
y_mail = np.array([0] * len(ham_emails) + [1] * len(spam_emails))

X_train_mail, X_test_mail, y_train_mail, y_test_mail = train_test_split(
    X_mail, y_mail, test_size=0.2, random_state=42)
print('Train:', len(X_train_mail), '| Test:', len(X_test_mail))

HTML in Text umwandeln und aus jeder E-Mail den Textinhalt holen.

In [ ]:
import re
from html import unescape

def html_to_plain_text(html):
    text = re.sub('<head.*?>.*?</head>', '', html, flags=re.M | re.S | re.I)
    text = re.sub(r'<a\s.*?>', ' HYPERLINK ', text, flags=re.M | re.S | re.I)
    text = re.sub('<.*?>', '', text, flags=re.M | re.S)
    text = re.sub(r'(\s*\n)+', '\n', text, flags=re.M | re.S)
    return unescape(text)

def email_to_text(email_obj):
    html = None
    for part in email_obj.walk():
        ctype = part.get_content_type()
        if ctype not in ('text/plain', 'text/html'):
            continue
        try:
            content = part.get_content()
        except Exception:
            content = str(part.get_payload())
        if ctype == 'text/plain':
            return content
        html = content
    if html is not None:
        return html_to_plain_text(html)

Optional `nltk` (Stemming) und `urlextract` (URL-Erkennung); fehlen sie, werden die Schritte übersprungen.

In [ ]:
try:
    import nltk
    stemmer = nltk.PorterStemmer()
except ImportError:
    stemmer = None
    print('nltk nicht gefunden – Stemming wird übersprungen.')

try:
    import urlextract
    url_extractor = urlextract.URLExtract()
except ImportError:
    url_extractor = None
    print('urlextract nicht gefunden – URL-Ersetzung wird übersprungen.')

Transformer 1: E-Mail → Wortzählung, mit Optionen für Kleinschreibung, Interpunktion, URL/NUMBER und Stemming.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class EmailToWordCounterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lower_case=True, remove_punctuation=True,
                 replace_urls=True, replace_numbers=True, stemming=True):
        self.lower_case = lower_case
        self.remove_punctuation = remove_punctuation
        self.replace_urls = replace_urls
        self.replace_numbers = replace_numbers
        self.stemming = stemming

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X_transformed = []
        for email_obj in X:
            text = email_to_text(email_obj) or ''
            if self.lower_case:
                text = text.lower()
            if self.replace_urls and url_extractor is not None:
                urls = sorted(set(url_extractor.find_urls(text)), key=len, reverse=True)
                for url in urls:
                    text = text.replace(url, ' URL ')
            if self.replace_numbers:
                text = re.sub(r'\d+(?:\.\d*)?(?:[eE][+-]?\d+)?', 'NUMBER', text)
            if self.remove_punctuation:
                text = re.sub(r'\W+', ' ', text, flags=re.M)
            word_counts = Counter(text.split())
            if self.stemming and stemmer is not None:
                stemmed_counts = Counter()
                for word, count in word_counts.items():
                    stemmed_counts[stemmer.stem(word)] += count
                word_counts = stemmed_counts
            X_transformed.append(word_counts)
        return np.array(X_transformed, dtype=object)

Transformer 2: Wortzählung → dünn besetzter Vektor der häufigsten Wörter (Spalte 0 = unbekannte Wörter).

In [ ]:
from scipy.sparse import csr_matrix

class WordCounterToVectorTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, vocabulary_size=1000):
        self.vocabulary_size = vocabulary_size

    def fit(self, X, y=None):
        total_count = Counter()
        for word_count in X:
            for word, count in word_count.items():
                total_count[word] += min(count, 10)
        most_common = total_count.most_common(self.vocabulary_size)
        self.vocabulary_ = {word: idx + 1 for idx, (word, _) in enumerate(most_common)}
        return self

    def transform(self, X, y=None):
        rows, cols, data = [], [], []
        for row, word_count in enumerate(X):
            for word, count in word_count.items():
                rows.append(row)
                cols.append(self.vocabulary_.get(word, 0))
                data.append(count)
        return csr_matrix((data, (rows, cols)),
                          shape=(len(X), self.vocabulary_size + 1))

In [ ]:
from sklearn.pipeline import Pipeline

preprocess_spam = Pipeline([
    ('email_to_wordcount', EmailToWordCounterTransformer()),
    ('wordcount_to_vector', WordCounterToVectorTransformer()),
])

X_train_transformed = preprocess_spam.fit_transform(X_train_mail)
print('Feature-Matrix:', X_train_transformed.shape)

Klassifikatoren per Kreuzvalidierung vergleichen.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

modelle = {
    'LogReg': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
}
for name, model in modelle.items():
    score = cross_val_score(model, X_train_transformed, y_train_mail, cv=3).mean()
    print('{:14s} CV-Genauigkeit: {:.4f}'.format(name, score))

Precision und Recall auf den Testdaten.

In [ ]:
from sklearn.metrics import precision_score, recall_score

X_test_transformed = preprocess_spam.transform(X_test_mail)

log_clf = LogisticRegression(max_iter=1000, random_state=42)
log_clf.fit(X_train_transformed, y_train_mail)
y_pred_mail = log_clf.predict(X_test_transformed)

print('Precision (Relevanz):    {:.2%}'.format(precision_score(y_test_mail, y_pred_mail)))
print('Recall (Sensitivität):   {:.2%}'.format(recall_score(y_test_mail, y_pred_mail)))

Die logistische Regression erreicht rund 95–99 % Precision und Recall.